## imports and helper function

In [ ]:
import os
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from cmdstanpy import CmdStanModel
import cmdstanpy
import numpy as np
from scipy.stats import invgamma
from scipy.stats import norm
from scipy.special import softmax
from gptools.stan import get_include

import pandas as pd
from glob import glob
from nteprsm import utils 
from cmdstanpy import stanfit
from settings import ROOT_DIR
import plotly.express as px
import utils as notebook_utils
# use customize plotly template
notebook_utils.set_custom_template()

import pickle
import seaborn as sns
import plotly.subplots as sp
from matplotlib import colors as mcolors

import arviz as az
from scipy.interpolate import CubicSpline

In [ ]:
NTEP_COLOR_SCALE = ["#8c510a","#bf812d","#dfc27d","#f6e8c3","#c7e9c0","#74c476","#41ab5d","#238b45","#00441b"]
def map_name2code(datahandler, column_name, code_column_name, invert=False):
    """
    Retrieves a dictionary mapping names to codes from specified columns.

    Args:
        column_name(str): The name of the column containing the names(
        e.g., 'ENTRY_NAME', 'RATER').
        code_column_name(str): The name of the column containing the codes
        (e.g., 'ENTRY_NAME_CODE', 'RATER_CODE').
        invert(bool): If True, returns a dictionary mapping codes to names.

    Returns:
        dict: Depending on 'invert', returns either a dict of {name: code}
        or {code: name}.
    """
    name2code = dict(datahandler.model_data.groupby(column_name)[code_column_name].first())

def rsm_probability(y, theta, thresholds):
    """
    Calculates the probability of a given class label in the model.

    Args:
    y (int): The class label for which the probability is calculated.
    theta (np.ndarray): An array of model parameters.
    tau (np.ndarry): The threshold parameters for the model.

    Returns:
    float: The probability of the given class label.

    """
    unsummed = np.concatenate(([0], theta - thresholds))
    probs = softmax(np.cumsum(unsummed))
    return probs[y]

def rsm_probability_vector(theta, thresholds):
    """
    Calculates the probability of set of class labels in given model

    Args:
    theta (np.ndarray): An array of model parameters.
    tau (np.ndarry): The threshold parameters for the model.

    Returns:
    np.ndarray: Array probability of given class label.
    """
    unsummed = np.concatenate(([0], theta - thresholds))
    probs = softmax(np.cumsum(unsummed))
    return probs
    
def plot_rater_characteristic_curve(
        betas,
        min_theta=-6,
        max_theta=6,
        resolution=500,
        colors=px.colors.diverging.Spectral,
        dimensions=None,
    ) -> go.Figure:
        """
        Plot the characteristic curves for raters based on the fitted Stan model.

        Args:
            rater_id (int, optional): The rater ID to plot. If None, all raters
                will be plotted. Defaults to None.
            dimensions (tuple, optional): Dimensions of the plot as (width, height).

        Returns:
            A Plotly figure object containing the plotted characteristic curves.
        """
        betas_with_bounds = np.concatenate(([min_theta], betas, [max_theta]))
        x = np.linspace(min_theta, max_theta, int((max_theta - min_theta) * resolution))
        num_categories = len(betas) + 1
        fig = go.Figure()
        fig.update_layout(
            template="ggplot2",
            xaxis_title="Turf Quality on Latent Scale",
            yaxis_title="Probability",
            legend=dict(x=1.02, y=1),
        )
        for i in range(num_categories):
            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=[rsm_probability(i, theta, betas) for theta in x],
                    line=dict(width=2, color=colors[i]),
                    name=str(i + 1),
                )
            )
            fig.add_shape(
                type="rect",
                x0=betas_with_bounds[i],
                x1=betas_with_bounds[i + 1],
                y0=1.02,
                y1=1.1,
                fillcolor=colors[i],
            )
            if i != num_categories - 1:
                fig.add_shape(
                    type="line",
                    x0=betas[i],
                    x1=betas[i],
                    y0=0,
                    y1=1,
                    line=dict(color=colors[i], dash="dot"),
                )
        if dimensions:
            fig.update_layout(width=dimensions[0], height=dimensions[1])
        return fig

def hex_to_rgba(hex_color, alpha=0.2):
    """Convert a HEX color (e.g., '#1f77b4') to an RGBA string with transparency."""
    rgb = mcolors.hex2color(hex_color)  # Convert hex to RGB (normalized 0-1)
    return f'rgba({int(rgb[0]*255)}, {int(rgb[1]*255)}, {int(rgb[2]*255)}, {alpha})'

def dms_to_decimal(coord):
    # Split the coordinate into components
    parts = coord.split()
    degrees = int(parts[0][:-1])  # Remove the "°"
    minutes = int(parts[1][:-1])  # Remove the "'"
    direction = parts[2]          # Direction (N, S, E, W)
    
    # Convert to decimal degrees
    decimal = degrees + (minutes / 60)
    
    # Adjust for direction
    if direction in ['S', 'W']:
        decimal = -decimal
    
    return decimal

In [ ]:
# Define colors for each entry
def plot_seasonality_curve(loc_code, entries, colors, df, ci = 0.95):
    #colors = [plt.cm.tab20(i) for i in range(20)]#['blue', 'green', 'red']
    fig = go.Figure()
    # time_effect_data_points
    filt_df =  df[df['test_loc_code'] == loc_code]
    rating_event_data = filt_df[['adj_time_of_year','rating_event_code']].drop_duplicates()
    data_points_time = rating_event_data['adj_time_of_year']
    data_points_rating_events = rating_event_data['rating_event_code']
    data_points_mean = fit.time_effect[:,loc_code-1,:,:].mean(axis=0)
    
    for i, entry in enumerate(entries):
        entry_data = [data_points_mean[entry-1][j-1] for j in data_points_rating_events]
        entry_name = entry_code2name[entry]
        legend_group = f"Entry {entry_name}"    
        samples = fit.pred_time_effect[:, loc_code-1, entry-1, :]  # Shape: (n_samples, n_time_points)
        
        # Compute statistics
        time_points = 2*np.arange(samples.shape[1])  # Time indices
        # time points really should be time_points + 2
        
        mean_values = np.mean(samples, axis=0)  # Mean over samples
        lower_bound = np.percentile(samples, 100 * (1 - ci) / 2, axis=0)  # 2.5th percentile
        upper_bound = np.percentile(samples, 100 * (ci + (1 - ci) / 2), axis=0)  # 97.5th percentile
    
        
        # wraparound adjustment
        mean_values =  [mean_values[-1]]+ list(mean_values[:-1])
        lower_bound = np.array([lower_bound[-1]] + list(lower_bound[:-1]))
        upper_bound = np.array([upper_bound[-1]] + list(upper_bound[:-1]))
        
        # Add data point
        
        fig.add_trace(go.Scatter(
            x=data_points_time * 100, y=entry_data,
            mode='markers',
            marker=dict(
                symbol='circle',  # Set the marker shape to "X"
                color=colors[i],  # Keep the color
                size=10  # Optionally, set the size of the marker
            ),
            name=f'{entry_name} Mean',
            legendgroup=legend_group,
            showlegend=False,
        ))
    
        # Add mean line
        fig.add_trace(go.Scatter(
            x=time_points, y=mean_values,
            mode='lines',
            line=dict(color=colors[i], width=2),
            name=f'{entry_name}',
            legendgroup=legend_group,
        ))
        
        # Add 95% Credible Interval (Shaded Region)
        fig.add_trace(go.Scatter(
            x=time_points.tolist() + time_points[::-1].tolist(),
            y=upper_bound.tolist() + lower_bound[::-1].tolist(),
            fill='toself',
            fillcolor=hex_to_rgba(colors[i], alpha=0.2)  ,
            line=dict(color='rgba(255,255,255,0)'),
            name=f'{entry_name} 95% CI',
            legendgroup=legend_group,  # Grouping
            showlegend=False,
        ))
    
    # Compute correct tick positions based on actual day of the year
    days_in_month = np.array([0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])  # Non-leap year
    cumulative_days = np.cumsum(days_in_month)+1  # Cumulative sum to get end of each month
    month_positions = 100*cumulative_days / 365  # Normalize to range 0-100
    month_positions[0] += 0.8
    #month_positions = np.insert(month_positions, 0, 0)
    # Generate month labels
    month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    fig.update_layout(
        xaxis=dict(
            tickvals=month_positions,  # Correctly spaced tick positions
            ticktext=month_labels,  # Month names
            title="Time of Year"
        ),
        yaxis_title="Seasonality (Latent Scale)",
        title=f"Turfgrass Seasonality at {testloc_code2name[loc_code]}, {ci} credible interval",
        template="ggplot2",
        legend=dict(title="Entries")
    )
    
    return fig

# Define colors for each entry
def compare_seasonality_curve_by_locs(loc_codes, entry, colors, df, ci = 0.95):
    #colors = [plt.cm.tab20(i) for i in range(20)]#['blue', 'green', 'red']
    fig = go.Figure()
    for i, loc_code in enumerate(loc_codes):
        # time_effect_data_points
        filt_df =  df[df['test_loc_code'] == loc_code]
        rating_event_data = filt_df[['adj_time_of_year','rating_event_code']].drop_duplicates()
        data_points_time = rating_event_data['adj_time_of_year']
        data_points_rating_events = rating_event_data['rating_event_code']
        data_points_mean = fit.time_effect[:,loc_code-1,:,:].mean(axis=0)
        entry_data = [data_points_mean[entry-1][j-1] for j in data_points_rating_events]
        entry_name = entry_code2name[entry]
        legend_group = f"{testloc_code2name[loc_code]}"    
        samples = fit.pred_time_effect[:, loc_code-1, entry-1, :]  # Shape: (n_samples, n_time_points)
        
        # Compute statistics
        time_points = 2*np.arange(samples.shape[1])  # Time indices
        # time points really should be time_points + 2
        
        mean_values = np.mean(samples, axis=0)  # Mean over samples
        lower_bound = np.percentile(samples, 100 * (1 - ci) / 2, axis=0)  # 2.5th percentile
        upper_bound = np.percentile(samples, 100 * (ci + (1 - ci) / 2), axis=0)  # 97.5th percentile
    
        
        # wraparound adjustment
        mean_values =  [mean_values[-1]]+ list(mean_values[:-1])
        lower_bound = np.array([lower_bound[-1]] + list(lower_bound[:-1]))
        upper_bound = np.array([upper_bound[-1]] + list(upper_bound[:-1]))
        
        # Add data point
        
        fig.add_trace(go.Scatter(
            x=data_points_time * 100, y=entry_data,
            mode='markers',
            marker=dict(
                symbol='circle',  # Set the marker shape to "X"
                color=colors[i],  # Keep the color
                size=10  # Optionally, set the size of the marker
            ),
            name=f'{legend_group} Mean',
            legendgroup=legend_group,
            showlegend=False,
        ))
    
        # Add mean line
        fig.add_trace(go.Scatter(
            x=time_points, y=mean_values,
            mode='lines',
            line=dict(color=colors[i], width=2),
            name=f'{legend_group}',
            legendgroup=legend_group,
        ))
        
        # Add 95% Credible Interval (Shaded Region)
        fig.add_trace(go.Scatter(
            x=time_points.tolist() + time_points[::-1].tolist(),
            y=upper_bound.tolist() + lower_bound[::-1].tolist(),
            fill='toself',
            fillcolor=hex_to_rgba(colors[i], alpha=0.2)  ,
            line=dict(color='rgba(255,255,255,0)'),
            name=f'{legend_group} 95% CI',
            legendgroup=legend_group,  # Grouping
            showlegend=False,
        ))
    
    # Compute correct tick positions based on actual day of the year
    days_in_month = np.array([0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])  # Non-leap year
    cumulative_days = np.cumsum(days_in_month)+1  # Cumulative sum to get end of each month
    month_positions = 100*cumulative_days / 365  # Normalize to range 0-100
    month_positions[0] += 0.8
    #month_positions = np.insert(month_positions, 0, 0)
    # Generate month labels
    month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    fig.update_layout(
        xaxis=dict(
            tickvals=month_positions,  # Correctly spaced tick positions
            ticktext=month_labels,  # Month names
            title="Time of Year"
        ),
        yaxis_title="Seasonality (Latent Scale)",
        title=f"Turfgrass Seasonality for {entry_code2name[entry]}, {ci} credible interval",
        template="ggplot2",
        legend=dict(title="Trial Locations")
    )
    
    return fig

# Define colors for each entry
def compare_seasonality_with_annual_model(entry, colors, df, ci = 0.95):
    #colors = [plt.cm.tab20(i) for i in range(20)]#['blue', 'green', 'red']
    fig = go.Figure()
    
    # time_effect_data_points
    filt_df =  df[df['test_loc_code'] == 1]
    rating_event_data = filt_df[['adj_time_of_year','rating_event_code']].drop_duplicates()
    data_points_time = rating_event_data['adj_time_of_year']
    data_points_rating_events = rating_event_data['rating_event_code']
    
    data_points_mean = fit.time_effect[:,0,:,:].mean(axis=0)
    entry_data = [data_points_mean[entry-1][j-1] for j in data_points_rating_events]
    
    entry_name = entry_code2name[entry]
    legend_group = "geographical"    
    samples = fit.pred_time_effect[:, 0, entry-1, :]  # Shape: (n_samples, n_time_points)
    
    # Compute statistics
    time_points = 2*np.arange(samples.shape[1])  # Time indices
    # time points really should be time_points + 2
    
    mean_values = np.mean(samples, axis=0)  # Mean over samples
    lower_bound = np.percentile(samples, 100 * (1 - ci) / 2, axis=0)  # 2.5th percentile
    upper_bound = np.percentile(samples, 100 * (ci + (1 - ci) / 2), axis=0)  # 97.5th percentile

    
    # wraparound adjustment
    mean_values =  [mean_values[-1]]+ list(mean_values[:-1])
    lower_bound = np.array([lower_bound[-1]] + list(lower_bound[:-1]))
    upper_bound = np.array([upper_bound[-1]] + list(upper_bound[:-1]))
    
    # Add data point
    
    fig.add_trace(go.Scatter(
        x=data_points_time * 100, y=entry_data,
        mode='markers',
        marker=dict(
            symbol='circle',  # Set the marker shape to "X"
            color=colors[0],  # Keep the color
            size=10  # Optionally, set the size of the marker
        ),
        name=f'Spatial coregionalization model data',
        legendgroup=legend_group,
        showlegend=False,
    ))

    # Add mean line
    fig.add_trace(go.Scatter(
        x=time_points, y=mean_values,
        mode='lines',
        line=dict(color=colors[0], width=2),
        name=f'Spatial coregionalization model',
        legendgroup=legend_group,
    ))
    
    # Add 95% Credible Interval (Shaded Region)
    fig.add_trace(go.Scatter(
        x=time_points.tolist() + time_points[::-1].tolist(),
        y=upper_bound.tolist() + lower_bound[::-1].tolist(),
        fill='toself',
        fillcolor=hex_to_rgba(colors[0], alpha=0.2)  ,
        line=dict(color='rgba(255,255,255,0)'),
        name=f'Spatial coregionalization model CI',
        legendgroup=legend_group,  # Grouping
        showlegend=False,
    ))
    
    ######
    ######
    # Plot original annual seasonality without geographical
    data_points = fit_as.time_effect
    legend_group = f"independent location"    
    samples = fit_as.pred_time_effect[:, entry-1, :]  # Shape: (n_samples, n_time_points)
    
    # Compute statistics
    time_points = np.arange(samples.shape[1]) + 1  # Time indices
    
    mean_values = np.mean(samples, axis=0)  # Mean over samples
    lower_bound = np.percentile(samples, 100 * (1 - ci) / 2, axis=0)  # 2.5th percentile
    upper_bound = np.percentile(samples, 100 * (ci + (1 - ci) / 2), axis=0)  # 97.5th percentile

    # wraparound adjustment
    time_points = np.array([0] + list(time_points)[:-2])
    mean_values =  [mean_values[-1]]+ list(mean_values[:-2])
    lower_bound = np.array([lower_bound[-1]] + list(lower_bound[:-2]))
    upper_bound = np.array([upper_bound[-1]] + list(upper_bound[:-2]))

    # Add data point
    fig.add_trace(go.Scatter(
        x=data_points_time * 100, y=data_points.mean(axis=0)[entry-1],
        mode='markers',
        marker=dict(
            symbol='circle',  # Set the marker shape to "X"
            color=colors[1],  # Keep the color
            size=10  # Optionally, set the size of the marker
        ),
        name=f'{entry_name} Mean',
        legendgroup=legend_group,
        showlegend=False,
    ))

    # Add mean line
    fig.add_trace(go.Scatter(
        x=time_points, y=mean_values,
        mode='lines',
        line=dict(color=colors[1], width=2),
        name=f'Independent locations',
        legendgroup=legend_group,
    ))
    
    # Add 95% Credible Interval (Shaded Region)
    fig.add_trace(go.Scatter(
        x=time_points.tolist() + time_points[::-1].tolist(),
        y=upper_bound.tolist() + lower_bound[::-1].tolist(),
        fill='toself',
        fillcolor=hex_to_rgba(colors[1], alpha=0.2)  ,
        line=dict(color='rgba(255,255,255,0)'),
        name=f'{entry_name} 95% CI',
        legendgroup=legend_group,  # Grouping
        showlegend=False,
    ))
    
    # Compute correct tick positions based on actual day of the year
    days_in_month = np.array([0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])  # Non-leap year
    cumulative_days = np.cumsum(days_in_month)+1  # Cumulative sum to get end of each month
    month_positions = 100*cumulative_days / 365  # Normalize to range 0-100
    month_positions[0] += 0.8
    #month_positions = np.insert(month_positions, 0, 0)
    # Generate month labels
    month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    fig.update_layout(
        xaxis=dict(
            tickvals=month_positions,  # Correctly spaced tick positions
            ticktext=month_labels,  # Month names
            title="Time of Year"
        ),
        yaxis_title="Seasonality (Latent Scale)",
        title=f"Turfgrass Seasonality for {entry_code2name[entry]}, {ci} credible interval",
        template="ggplot2",
        legend=dict(title="Models")
    )
    
    return fig

## get data

In [ ]:
ddfs = []
plt_layouts = []
rowcolswaps = []

files = ['quality_in1', 'quality_mi1', 'quality_mn1', 'quality_nj2', 'quality_ok1', 'quality_ut1', 'quality_nc1']
loc_counter = 0

for file in files:
    df = pd.read_csv(f'data/raw/{file}.csv')
    try:
        ## create adj_time_of_year columns
        df.columns = [col.lower() for col in df.columns]
        df['date']=pd.to_datetime(df["date"], format='%m/%d/%y')
        df["adj_day_of_year"] = df.date.dt.day_of_year - (
            df.date.dt.is_leap_year * df.date.dt.day_of_year >= 60
        )
        df["adj_time_of_year"] = df.adj_day_of_year / 365
        
        ## quality_ok1 uses ploc_code
        if 'ploc_code' in df.columns:
            df['plt_id'] = df['ploc_code']

        if df['col'].max() > df['row'].max():
            rowcolswaps.append(1)
            temp = df['col'].values
            df['col'] = df['row']
            df['row'] = temp
        else:
            rowcolswaps.append(0)
        df['plt_id_code'] = pd.Categorical(df["plt_id"]).codes + 1
        df["entry_cumcount"] = df.groupby("entry_name").cumcount() + 1
        df["file"] = file
        df["trial_location_code"] = loc_counter
        loc_counter +=1
        dfs.append(df)

        # handle plot_data
        plt_data = df[['plt_id_code', 'row', 'col']].drop_duplicates().sort_values('plt_id_code')
        assert len(plt_data) == int(plt_data.plt_id_code.nunique())
        plt_layouts.append(plt_data)
        
    except Exception as e:
        print(f"Error processing {file}")
        print(f"Error message: {e}")
        print(f"df columns: {df.columns}")

In [ ]:
df = pd.concat(dfs, axis=0)

df = df.assign(
    entry_name_code=pd.Categorical(df["entry_name"]).codes + 1,
    rater_code=pd.Categorical(df["rater"]).codes + 1,
    rating_event_code=pd.Categorical(df["rating_event"]).codes + 1,
    test_loc_code=pd.Categorical(df["test_loc"]).codes + 1,
)

df[['entry_name','test_loc']].drop_duplicates().sort_values('entry_name')

In [ ]:
entry_name2code = dict(df.groupby('entry_name')['entry_name_code'].first())
entry_code2name = dict(df.groupby('entry_name_code')['entry_name'].first())

testloc_name2code = dict(df.groupby('test_loc')['test_loc_code'].first())
testloc_code2name =  dict(df.groupby('test_loc_code')['test_loc'].first())

In [ ]:
rowcolswaps

## Geographical locations

In [ ]:
df[['test_loc', 'test_loc_code']].drop_duplicates()

In [ ]:
locs = df['test_loc'].unique()
locs

In [ ]:
#test_loc_to_code = df[['test_loc', 'test_loc_code']].drop_duplicates().set_index('test_loc')['test_loc_code'].to_dict()
#code_to_test_loc = df[['test_loc', 'test_loc_code']].drop_duplicates().set_index('test_loc_code')['test_loc'].to_dict()

coordinates = {
    'West Lafayette, IN': ("040º 26' N", "086° 55' W"),
    'East Lansing, MI': ("042º 42' N", "084° 28' W"),
    'St. Paul, MN': ("44º 53' N", "93° 13' W"),
    'Raleigh, NC': ("035º 52' N", "078° 47' W"),
    'Adelphia, NJ': ("040º 29' N", "074° 27' W"),
    'Stillwater, OK': ("038º 02' N", "084° 36' W"),
    'Logan, UT': ("041º 47' N", "111° 51' W")
}

lats = []
lons = []

for i in range(len(locs)):
    loc = testloc_code2name[i+1]
    lat, lon = coordinates[loc]
    decimal_lat = dms_to_decimal(lat)
    decimal_lon = dms_to_decimal(lon)
    lats.append(decimal_lat)
    lons.append(decimal_lon)

In [ ]:
# Data
locations = df['test_loc'].unique()
plt.figure(figsize=(10, 6))
plt.scatter(lons, lats, c='blue', marker='o', label='Locations')

# Annotate points
for i, location in enumerate(locs):
    plt.text(lons[i] + 0.3, lats[i] + 0.2, location, fontsize=9)  # slight offset

# Add axis labels and title
plt.title('Geographical Locations', fontsize=14)
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)

# Add grid and legend
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right')

# Add margins to avoid overlap
lon_min, lon_max = min(lons), max(lons)
lat_min, lat_max = min(lats), max(lats)
plt.xlim(lon_min - 2, lon_max + 2)   # Add margin to x-axis
plt.ylim(lat_min - 1, lat_max + 1)   # Add margin to y-axis

plt.tight_layout()
plt.show()

### Model sampling

In [ ]:
target = 'quality'
padding = 1
pred_N = 50
M_f = 8
iter_warmpup = 350
iter_sampling = 500
num_chains = 2

In [ ]:
target = 'quality'
padding = 1
pred_N = 50
M_f = 8

row_list = [plt_layouts[i].sort_values('plt_id_code')['row'].values for i in range(df['test_loc_code'].nunique())]
col_list = [plt_layouts[i].sort_values('plt_id_code')['col'].values for i in range(df['test_loc_code'].nunique())]

stan_data = {
    "N": len(df[target]),
    "L": df['test_loc_code'].nunique(),
    "num_categories": int(df[target].nunique()),
    "y": (df['quality'] - df[target].min()).values,
    "num_raters": int(df.rater.nunique()),
    "num_entries": int(df.entry_name.nunique()),
    "trial_loc": df['test_loc_code'],

    # trial layouts
    "num_plots": df.groupby('test_loc_code')['plt_id_code'].max().values,
    'total_num_plots':np.sum(df.groupby('test_loc_code')['plt_id_code'].max().values),
    "num_rows": df.groupby('test_loc_code')['row'].max().values,
    "num_cols": df.groupby('test_loc_code')['col'].max().values,
    "plot_row": np.concatenate(row_list),
    "plot_col": np.concatenate(col_list),
    "rater_id": df.rater_code.values,
    "entry_id": df.entry_name_code.values,
    "plot_id": df.plt_id_code.values,

    # geographical data
    "lats":lats,
    "lons":lons,

    ## time
    "num_rating_events": int(df['rating_event_code'].nunique()),
    "rating_event_codes": df['rating_event_code'],
    "time": df[['rating_event_code','adj_time_of_year']].drop_duplicates().sort_values('rating_event_code').set_index('rating_event_code').values.reshape(-1),

    # maps rating event -> temperature
    ## predictions
    'pred_N':pred_N,

    ## approximation params
    "padding": padding,
    "M_f":M_f,
}


In [ ]:
# load model configuration
config_file = ROOT_DIR/"config/nteprsm_njkbg07.yml"
config = utils.load_config(config_file)
config["sampling"]['save_warmup'] = False
config["sampling"]["iter_sampling"] = iter_sampling
config["sampling"]["iter_warmup"] = iter_warmpup
config["sampling"]["parallel_chains"] = num_chains
config

In [ ]:
## non hiearchical raters
nteprsm = CmdStanModel(
    stan_file='models/geographical_model.stan',
    stanc_options={"include-paths": get_include()},
)
fit = nteprsm.sample(data=stan_data, **config["sampling"])

In [ ]:
## hiearchical raters
nteprsm = CmdStanModel(
    stan_file='models/geographical_model_hierarchical_rater.stan',
    stanc_options={"include-paths": get_include()},
)
fit = nteprsm.sample(data=stan_data, **config["sampling"])

In [ ]:
plt.plot(fit.lengthscale_geo)

In [ ]:
import pickle

with open("geographical_scale_alignment_hrater_23_apr_2025.pkl","wb") as file:
    pickle.dump(fit, file)

## Posterior sample analysis

In [ ]:
## Load new jersey sampled dataset for comparison
with open('annual_seasonality_nj2.pkl', 'rb') as file:
    fit_as = pickle.load(file)

In [ ]:
colors = ['royalblue', 'mediumseagreen', 'deepskyblue', 'crimson']

entries = [i for i in range(len(entry_code2name))]

entries_str = ['Kenblue', 'Skye', 'Selway', 'Pivot']
entries = [entry_name2code[w] for w in entries_str]

In [ ]:
compare_seasonality_curve_by_locs([1,2,3,4,], 58, colors, df, 0.95)

In [ ]:
compare_seasonality_curve_by_locs([1,2,3,4,], 84, colors, df, 0.95)

In [ ]:
plot_seasonality_curve(1, entries, colors, df, 0.95)

In [ ]:
df_entry = df[df['entry_name'] == 'Babe']
df_entry.groupby('test_loc')['quality'].mean()

In [ ]:
df_entry = df[df['entry_name'] == 'Selway']
df_entry.groupby('test_loc')['quality'].mean()

In [ ]:
df_entry = df[df['entry_name'] == 'Pivot']
df_entry.groupby('test_loc')['quality'].mean()

In [ ]:
df_entry = df[df['entry_name'] == 'Blue Devil']
df_entry.groupby('test_loc')['quality'].mean()

In [ ]:
plot_seasonality_curve(2, entries, colors, df, 0.95)

In [ ]:
plot_seasonality_curve(3, entries, colors, df, 0.95)

In [ ]:
plot_seasonality_curve(4, entries, colors, df, 0.95)

In [ ]:
compare_seasonality_with_annual_model(58, colors, df)

In [ ]:
compare_seasonality_with_annual_model(84, colors, df)

In [ ]:
entries

In [ ]:
plot_seasonality_curve(1, entries, colors, df, 0.95)

In [ ]:
entries

In [ ]:
compare_seasonality_curve_by_locs([1,2,3,4,], 58, ["red", "#4fa3d1", "orange", "#2b7b93"], df, 0.95)

In [ ]:
compare_seasonality_curve_by_locs([1,2,3,4,], 78, ["red", "#4fa3d1", "orange", "#2b7b93"], df, 0.95)

In [ ]:
compare_seasonality_with_annual_model(78, colors, df)

In [ ]:
entry_name2code['Blue Devil']

In [ ]:
compare_seasonality_curve_by_locs([1,2,3,4,], 26, ["red", "#4fa3d1", "orange", "#2b7b93"], df, 0.95)

In [ ]:
compare_seasonality_with_annual_model(26, colors, df)

In [ ]:
compare_seasonality_curve_by_locs([1,2,3,4,], 29, ["red", "#4fa3d1", "orange", "#2b7b93"], df, 0.95)

In [ ]:
compare_seasonality_with_annual_model(29, colors, df)